In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import iirnotch, filtfilt, butter
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [ ]:
data = pd.read_csv(r"D:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv")
data.columns = data.columns.str.strip()
print(data)
print(data.columns)

In [ ]:
kolom_nan = ['ERP_Memory_Recall', 'ERP_Arithmetic', 'ERP_Visual_Pattern']

data_bersih = data.drop(columns=kolom_nan)

print("--- Info Data Setelah Bersih dari NaN ---")
data_bersih.info()

print("\n--- Data Head ---")
print(data_bersih.head())

In [ ]:
# 'y' adalah apa yang ingin kita prediksi
y = data_bersih['Target_Label']

# 'X' adalah semua kolom lain yang kita gunakan untuk memprediksi
X = data_bersih.drop(columns=['Target_Label'])

print("Data X (fitur) sebelum diubah:")
print(X.head())

print("\nData y (target):")
print(y.head())

In [ ]:
print(y)

In [ ]:
# Terapkan One-Hot Encoding pada kolom 'Task'
X_encoded = pd.get_dummies(X, columns=['Task'])

print("--- Data Info (Setelah 'Task' Di-encode) ---")
# Anda akan lihat ada kolom baru: Task_Resting State, Task_Memory Recall, dll.
X_encoded.info()

In [ ]:
# 1. Tentukan kolom mana saja yang perlu di-scale
# (Yaitu semua kolom yang bertipe angka)
kolom_untuk_scale = X_encoded.select_dtypes(include='number').columns

# 2. Buat scaler-nya
scaler = StandardScaler()

# 3. Terapkan scaler ke data X
X_scaled = X_encoded.copy() # Buat salinan
X_scaled[kolom_untuk_scale] = scaler.fit_transform(X_scaled[kolom_untuk_scale])

print("\n--- Data Head (Setelah Scaling) ---")
print(X_scaled.head())

In [ ]:
# Import 'train_test_split' jika Anda belum (jika Sel [1] lupa dijalankan)
from sklearn.model_selection import train_test_split

# 'X_scaled' adalah fitur Anda yang sudah siap
# 'y' adalah target Anda (dari Sel [3])

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, 
                                                    test_size=0.4,    # 20% data untuk testing
                                                    random_state=42,  # Agar hasil 'acak'-nya selalu sama
                                                    stratify=y)       # Memastikan proporsi label seimbang

print("--- Data Berhasil Dibagi ---")
print(f"Jumlah data Training: {X_train.shape[0]} baris ({X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])*100:.1f}%)")
print(f"Jumlah data Testing:  {X_test.shape[0]} baris ({X_test.shape[0]/(X_train.shape[0]+X_test.shape[0])*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# (Ini berasumsi 'data_bersih' sudah ada dari Sel [2])
try:
    # 1. Pilih hanya kolom 'Task' dan kolom-kolom PSD
    psd_columns = [col for col in data_bersih.columns if 'PSD' in col]
    plot_data = data_bersih[['Task'] + psd_columns]

    # 2. Hitung rata-rata semua kolom PSD, dikelompokkan berdasarkan 'Task'
    avg_psd_by_task = plot_data.groupby('Task').mean()

    # 3. Buat plot
    plt.figure(figsize=(15, 7))
    
    # Kita Transpose (T) DataFrame agar Task menjadi garis, dan Channel menjadi sumbu X
    # 'ax=plt.gca()' dipakai agar plotnya menggunakan ukuran figure (figsize) yang kita set
    avg_psd_by_task.T.plot(ax=plt.gca(), linewidth=2)
    
    plt.title('Rata-rata PSD (64 Channel) per Tugas Kognitif')
    plt.xlabel('Channel PSD')
    plt.ylabel('Rata-rata Power Spectral Density (PSD)')
    
    # 4. Merapikan Sumbu X (karena 64 label channel terlalu banyak)
    # Kita hanya akan menampilkan label setiap 5 channel
    ticks_to_show = list(range(0, 64, 5)) # Tampilkan tick di channel 0, 5, 10...
    labels = [f'Ch {i+1}' for i in ticks_to_show] # Beri label 'Ch 1', 'Ch 6', ...
    
    plt.xticks(ticks=ticks_to_show, labels=labels, rotation=45)
    
    plt.legend(title='Tugas Kognitif')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Simpan gambar atau tampilkan
    plt.savefig('psd_line_chart.png')
    # plt.show() # Nyalakan ini jika plot tidak muncul otomatis

    print("Plot line chart 'psd_line_chart.png' berhasil disimpan.")

except NameError:
    print("Error: Variabel 'data_bersih' tidak ditemukan.")
    print("Pastikan Anda sudah menjalankan Sel [2] terlebih dahulu.")
except Exception as e:
    print(f"Terjadi error: {e}")

In [ ]:
file_path = r"D:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv"

try:
    df = pd.read_csv(file_path)
    print("File berhasil dimuat.")
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di '{file_path}'")
    raise

print("\nData Head (Data Asli):")
df.head()

In [ ]:
all_psd_cols = [col for col in df.columns if col.startswith('Channel_')]

id_columns = [col for col in df.columns if not col.startswith('Channel_')]

print(f"Total kolom PSD: {len(all_psd_cols)}")
print(f"Kolom ID: {id_columns}")

In [ ]:
# Buat list kosong untuk setiap pita frekuensi
delta_cols = []
theta_cols = []
alpha_cols = []
beta_cols = []
gamma_cols = []

# Loop melalui setiap kolom PSD
for col in all_psd_cols:
    try:
        # Ekstrak angka frekuensi dari nama kolom
        # e.g., 'Channel_15_PSD' -> 15
        freq = int(col.split('_')[1])

        # Kelompokkan berdasarkan spesifikasi
        # (<= 4 Hz)
        if freq <= 4:
            delta_cols.append(col)
        # (> 4 Hz dan <= 8 Hz)
        elif 4 < freq <= 8:
            theta_cols.append(col)
        # (> 8 Hz dan <= 13 Hz)
        elif 8 < freq <= 13:
            alpha_cols.append(col)
        # (> 13 Hz dan <= 30 Hz)
        elif 13 < freq <= 30:
            beta_cols.append(col)
        # (> 30 Hz)
        elif freq > 30:
            gamma_cols.append(col)
            
    except (ValueError, IndexError):
        # Abaikan kolom jika formatnya tidak 'Channel_X_PSD'
        print(f"Mengabaikan kolom: {col}")

# Cetak ringkasan pengelompokan
print(f"Delta (<= 4 Hz): {len(delta_cols)} kolom (Contoh: {delta_cols[:2]}...)")
print(f"Theta (5-8 Hz): {len(theta_cols)} kolom (Contoh: {theta_cols[:2]}...)")
print(f"Alpha (9-13 Hz): {len(alpha_cols)} kolom (Contoh: {alpha_cols[:2]}...)")
print(f"Beta (14-30 Hz): {len(beta_cols)} kolom (Contoh: {beta_cols[:2]}...)")
print(f"Gamma (> 30 Hz): {len(gamma_cols)} kolom (Contoh: {gamma_cols[:2]}...)")

In [ ]:
# Mulai DataFrame baru dengan kolom ID
df_bands = df[id_columns].copy()

# Hitung rata-rata power untuk setiap band
# axis=1 menghitung rata-rata per baris

# Cek jika list tidak kosong sebelum menghitung
if delta_cols:
    df_bands['delta_power'] = df[delta_cols].mean(axis=1)
if theta_cols:
    df_bands['theta_power'] = df[theta_cols].mean(axis=1)
if alpha_cols:
    df_bands['alpha_power'] = df[alpha_cols].mean(axis=1)
if beta_cols:
    df_bands['beta_power'] = df[beta_cols].mean(axis=1)
if gamma_cols:
    df_bands['gamma_power'] = df[gamma_cols].mean(axis=1)

# Tampilkan hasil DataFrame baru
print("DataFrame dengan Band Power Baru:")
df_bands.head()


In [ ]:
import altair as alt

In [ ]:
power_columns = [col for col in df_bands.columns if '_power' in col]

overall_power_means = df_bands[power_columns].mean()

df_plot_data = overall_power_means.reset_index()

df_plot_data.columns = ['Band', 'Power']

df_plot_data['Band'] = df_plot_data['Band'].str.replace('_power', '').str.capitalize()

band_order = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']

print("Data untuk plot (rata-rata keseluruhan):")
print(df_plot_data)

In [ ]:
simple_bar_chart = alt.Chart(df_plot_data).mark_bar().encode(
    x=alt.X('Band', sort=band_order, title='Pita Frekuensi'),
    
    y=alt.Y('Power', title='Rata-rata Power (PSD) Keseluruhan'),
    
    color=alt.Color('Band', legend=None),
    
    tooltip=['Band', 'Power']
).properties(
    title='Rata-rata Power Keseluruhan per Pita Frekuensi'
).interactive()

In [ ]:
simple_bar_chart.save('overall_average_bar_chart.json')
print("Grafik disimpan ke 'overall_average_bar_chart.json'")

simple_bar_chart

RESTING STATE

In [ ]:
resting = df[df['Task'] == 'Resting State']
resting_band_power = {
    'Delta': resting[delta_cols].mean().mean(),
    'Theta': resting[theta_cols].mean().mean(),
    'Alpha': resting[alpha_cols].mean().mean(),
    'Beta' : resting[beta_cols].mean().mean(),
    'Gamma': resting[gamma_cols].mean().mean()
}

print("Band Power - Resting State")
print(resting_band_power)

# Plot bar chart
plt.figure(figsize=(7, 5))
plt.bar(resting_band_power.keys(), resting_band_power.values())
plt.title("Band Power - Resting State")
plt.xlabel("Frequency Band")
plt.ylabel("Average Power (PSD)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


Memory Recall

In [ ]:
# Filter data kondisi
memory = df[df['Task'] == 'Memory Recall']

# Hitung band power
memory_band_power = {
    'Delta': memory[delta_cols].mean().mean(),
    'Theta': memory[theta_cols].mean().mean(),
    'Alpha': memory[alpha_cols].mean().mean(),
    'Beta' : memory[beta_cols].mean().mean(),
    'Gamma': memory[gamma_cols].mean().mean()
}

print("Band Power - Memory Recall")
print(memory_band_power)

# Plot bar chart
plt.figure(figsize=(7, 5))
plt.bar(memory_band_power.keys(), memory_band_power.values())
plt.title("Band Power - Memory Recall")
plt.xlabel("Frequency Band")
plt.ylabel("Average Power (PSD)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


Arithmetic Calculation

In [ ]:
# Filter data kondisi
arith = df[df['Task'] == 'Arithmetic Calculation']

# Hitung band power
arith_band_power = {
    'Delta': arith[delta_cols].mean().mean(),
    'Theta': arith[theta_cols].mean().mean(),
    'Alpha': arith[alpha_cols].mean().mean(),
    'Beta' : arith[beta_cols].mean().mean(),
    'Gamma': arith[gamma_cols].mean().mean()
}

print("Band Power - Arithmetic Calculation")
print(arith_band_power)

# Plot bar chart
plt.figure(figsize=(7, 5))
plt.bar(arith_band_power.keys(), arith_band_power.values())
plt.title("Band Power - Arithmetic Calculation")
plt.xlabel("Frequency Band")
plt.ylabel("Average Power (PSD)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


Visual Pattern Recognition

In [ ]:
# Filter data kondisi
visual = df[df['Task'] == 'Visual Pattern Recognition']

# Hitung band power
visual_band_power = {
    'Delta': visual[delta_cols].mean().mean(),
    'Theta': visual[theta_cols].mean().mean(),
    'Alpha': visual[alpha_cols].mean().mean(),
    'Beta' : visual[beta_cols].mean().mean(),
    'Gamma': visual[gamma_cols].mean().mean()
}

print("Band Power - Visual Pattern Recognition")
print(visual_band_power)

# Plot bar chart
plt.figure(figsize=(7, 5))
plt.bar(visual_band_power.keys(), visual_band_power.values())
plt.title("Band Power - Visual Pattern Recognition")
plt.xlabel("Frequency Band")
plt.ylabel("Average Power (PSD)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


tambahkan ke eksport e bntuk h5 dari prep ini 

In [ ]:
import pickle

In [ ]:
# Simpan semua data ke 1 file
data_preprocessed = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'scaler': scaler,
    'feature_names': X_train.columns.tolist()
}

# Simpan ke file
with open('eeg_preprocessed.pkl', 'wb') as f:
    pickle.dump(data_preprocessed, f)

print("✅ Data preprocessing berhasil disimpan ke 'eeg_preprocessed.pkl'")

# Cek ukuran file
import os
file_size = os.path.getsize('eeg_preprocessed.pkl') / (1024 * 1024)  # MB
print(f"📦 Ukuran file: {file_size:.2f} MB")

In [ ]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False, header=['Target_Label'])
y_test.to_csv('y_test.csv', index=False, header=['Target_Label'])

print("✅ Data juga disimpan ke CSV (X_train.csv, X_test.csv, dll)")

In [ ]:
import joblib
joblib.dump(scaler, 'scaler.pkl')
print("✅ Scaler disimpan ke 'scaler.pkl'")